<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" width="400">

# PDF Invoice Generation from Multiple Data Sources Practice Notebook

**Main Tutorial Link:** [Generate PDF Invoices in Python with ReportLab — Complete Guide](https://www.plus2net.com/python/pdf-invoice.php)

---

## 1. Environment Setup & Download Sample Datasets

Install `reportlab`, `sqlalchemy`, `openpyxl`, and `pandas` inside Google Colab and set up sample project directories.

In [ ]:
!pip install reportlab sqlalchemy openpyxl pandas > /dev/null
!mkdir -p output data

# Download sample student/sales datasets
!wget -q -O data/student.csv https://www.plus2net.com/python/download/student.csv
!wget -q -O data/student.xlsx https://www.plus2net.com/python/download/student.xlsx
!wget -q -O data/student.json https://www.plus2net.com/python/download/student.json
!wget -q -O data/student.xml https://www.plus2net.com/python/download/student.xml
!wget -q -O data/student.sql https://www.plus2net.com/python/download/student.sql

print("ReportLab environment initialized and all multi-source sample files ready!")

## 2. Define Blank Invoice Layout Template (`temp_invoice.py`)

Define `my_temp(c)` function to draw structural header lines, company header details, automated dates, vertical table gridlines, watermarks, and footer totals labels.

In [ ]:
from reportlab.lib.units import inch
from datetime import date

def my_temp(c):
    c.translate(inch, inch)
    c.setFont("Helvetica", 14)
    c.setStrokeColorRGB(0.1, 0.8, 0.1)
    c.setFillColorRGB(0, 0, 1)
    
    c.drawString(0, 9 * inch, "Shop No : 1234, ABCD Road")
    c.drawString(0, 8.7 * inch, "City Name: Mycity, ZIP : 12345")
    c.setFillColorRGB(0, 0, 0)
    c.line(0, 8.6 * inch, 6.8 * inch, 8.6 * inch)
    
    c.drawString(5.6 * inch, 9.5 * inch, 'Bill No :# 1234')
    dt = date.today().strftime('%d-%b-%Y')
    c.drawString(5.6 * inch, 9.3 * inch, dt)
    
    c.setFont("Helvetica", 8)
    c.drawString(3 * inch, 9.6 * inch, 'Tax No :# ABC1234')
    c.setFillColorRGB(1, 0, 0)
    c.setFont("Times-Bold", 40)
    c.drawString(4.3 * inch, 8.7 * inch, 'INVOICE')
    
    # Watermark
    c.rotate(45)
    c.setFillColorCMYK(0, 0, 0, 0.08)
    c.setFont("Helvetica", 140)
    c.drawString(2 * inch, 1 * inch, "SAMPLE")
    c.rotate(-45)
    
    # Table Column Titles
    c.setFillColorRGB(0, 0, 0)
    c.setFont("Times-Roman", 22)
    c.drawString(0.5 * inch, 8.3 * inch, 'Products')
    c.drawString(4 * inch, 8.3 * inch, 'Price')
    c.drawString(5 * inch, 8.3 * inch, 'Quantity')
    c.drawString(6.1 * inch, 8.3 * inch, 'Total')
    
    # Vertical Gridlines
    c.setStrokeColorCMYK(0, 0, 0, 1)
    c.line(3.9 * inch, 8.3 * inch, 3.9 * inch, 2.7 * inch)
    c.line(4.9 * inch, 8.3 * inch, 4.9 * inch, 2.7 * inch)
    c.line(6.1 * inch, 8.3 * inch, 6.1 * inch, 2.7 * inch)
    c.line(0.01 * inch, 2.5 * inch, 7 * inch, 2.5 * inch)
    
    # Footer Labels
    c.drawString(1 * inch, 1.8 * inch, 'Discount')
    c.drawString(1 * inch, 1.2 * inch, 'Tax')
    c.setFont("Times-Bold", 22)
    c.drawString(2 * inch, 0.8 * inch, 'Total')
    c.setFont("Times-Roman", 22)
    c.drawString(5.6 * inch, -0.1 * inch, 'Signature')
    c.setStrokeColorRGB(0.1, 0.8, 0.1)
    c.line(0, -0.7 * inch, 6.8 * inch, -0.7 * inch)
    
    c.setFont("Helvetica", 8)
    c.setFillColorRGB(1, 0, 0)
    c.drawString(0, -0.9 * inch, "\u00A9 plus2net.com")
    
    return c

print("Invoice template initialized!")

## 3. Generate Sales Invoice from Standard Dictionaries (`invoice_data.py`)

Iterate over sample sales dictionaries, compute sub-totals, calculate percentage-based discounts and taxes, and output the PDF invoice.

In [ ]:
from pathlib import Path
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

output_dir = Path('output')
pdf_invoice = output_dir / 'sales_invoice.pdf'

my_prod = {
    1: ['Hard Disk', 80], 2: ['RAM', 90], 3: ['Monitor', 75],
    4: ['CPU', 55], 5: ['Keyboard', 20], 6: ['Mouse', 10],
    7: ['Mother board', 50], 8: ['Power Supply', 20],
    9: ['Speaker', 50], 10: ['Microphone', 45]
}
my_sale = {1: 2, 3: 2, 7: 1, 4: 3, 6: 5, 5: 3, 2: 1, 9: 1, 10: 3}
discount_rate = 10
tax_rate = 12

c = canvas.Canvas(str(pdf_invoice), pagesize=letter)
c = my_temp(c)

c.setFillColorRGB(0, 0, 1)
c.setFont("Helvetica", 20)
row_gap = 0.6
line_y = 7.9
total = 0

for items in my_sale:
    c.drawString(0.1 * inch, line_y * inch, str(my_prod[items][0]))
    c.drawRightString(4.5 * inch, line_y * inch, str(my_prod[items][1]))
    c.drawRightString(5.5 * inch, line_y * inch, str(my_sale[items]))
    sub_total = my_prod[items][1] * my_sale[items]
    c.drawRightString(7 * inch, line_y * inch, str(sub_total))
    total = round(total + sub_total, 1)
    line_y = line_y - row_gap

c.drawRightString(7 * inch, 2.1 * inch, str(float(total)))
discount = round((discount_rate / 100) * total, 1)
c.drawRightString(4 * inch, 1.8 * inch, str(discount_rate) + '%')
c.drawRightString(7 * inch, 1.8 * inch, '-' + str(discount))

tax = round((tax_rate / 100) * (total - discount), 1)
c.drawRightString(4 * inch, 1.2 * inch, str(tax_rate) + '%')
c.drawRightString(7 * inch, 1.2 * inch, str(tax))

total_final = total - discount + tax
c.setFont("Times-Bold", 22)
c.setFillColorRGB(1, 0, 0)
c.drawRightString(7 * inch, 0.8 * inch, str(total_final))

c.showPage()
c.save()
print(f"Sales Invoice PDF created: {pdf_invoice}")

---
### Visit [www.plus2net.com](https://www.plus2net.com) for more Python ReportLab tutorials and downloadable source code.